# 01 Explore the cleaned social housing EPC table

Run `python -m housing_epc.pipeline` first. It writes the cleaned modelling table to `data/processed/`.

This notebook answers: what does the social housing stock in the data look like, and where is the below-C share highest?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from housing_epc import config
from housing_epc.features import age_band_label

df = pd.read_csv(config.PROCESSED_DIR / "social_homes_features.csv.gz")
df["build_era"] = age_band_label(df["construction_year"])
print(f"{len(df):,} homes, {df['local_authority'].nunique()} local authorities")
df.head()

## Data quality: how complete is each feature?

In [ ]:
(df.isna().mean().sort_values(ascending=False) * 100).round(1).to_frame("% missing")

## Rating distribution

In [ ]:
order = list("ABCDEFG")
counts = df["current_energy_rating"].value_counts().reindex(order, fill_value=0)
ax = counts.plot.bar(color=["#2a78d6" if r in "ABC" else "#eb6834" for r in order], rot=0)
ax.set_title("Social homes by current EPC rating", loc="left")
ax.set_ylabel("Homes")
ax.spines[["top", "right"]].set_visible(False)
print(f"Below C: {df['below_c'].mean():.1%}")

## Where is the below-C share highest?

In [ ]:
for col in ["build_era", "property_type", "fuel_group", "heating_system", "wall_insulation"]:
    summary = (df.groupby(col)["below_c"].agg(homes="size", below_c="mean")
                 .query("homes >= 50").sort_values("below_c", ascending=False))
    display(summary.style.format({"below_c": "{:.0%}", "homes": "{:,}"}).set_caption(col))

## Interaction: build era by property type

In [ ]:
pivot = df.pivot_table(index="build_era", columns="property_type", values="below_c", aggfunc="mean")
pivot = pivot.reindex(["pre-1930", "1930-49", "1950-66", "1967-82", "1983-2002", "2003+"])
pivot.style.format("{:.0%}").background_gradient(cmap="Blues", axis=None)

## Notes for the write-up

- Which archetypes stand out?
- Which features have too much missing data to rely on?
- Does anything look like a data quality issue rather than a real pattern?